"""
========================================================================
NOTEBOOK 02: GPT MAIN EXPERIMENT
========================================================================

Loads outputs from Notebook 01 and runs the full prompt-hierarchy probe
on GPT-4o-mini.

Design:
  - 4 prompts: A (current/reconstruction), B (minimal anchor),
               D (anti-brochure), E (frame + anti-brochure)
  - 20 headlines × 3 reps × 4 prompts = 240 GPT-4o-mini calls
  - Estimated cost: ~$6
  - Estimated time: 30-60 min (with checkpointing — restart-safe)

Outputs (data/generations/, data/results/):
  - generations_gpt.csv               # all 240 generated articles
  - axis1_gpt.csv                     # entity scores per article
  - axis2_gpt.csv                     # sentiment scores per article
  - results_main_summary.csv          # aggregate by prompt × desk
  - results_robustness_length.csv     # length confound check
  - results_headline_aggregated.csv   # pseudo-replication-corrected stats

Prereqs:
  - Notebook 01 has been run successfully
  - OPENAI_API_KEY is set in os.environ
"""

In [ ]:

# =============================================================================
# CELL 1: Setup
# =============================================================================

import os
import re
import json
import gc
import time
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Apple Silicon stability
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# Suppress the >512-token tokenizer warning we already analyzed
warnings.filterwarnings("ignore", message="Token indices sequence length")

# Paths — relative to this notebook (notebooks/), same convention as Notebook 01
ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
GENERATIONS = ROOT / "data" / "generations"
RESULTS = ROOT / "data" / "results"

GENERATIONS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

print(f"ROOT:        {ROOT}")
print(f"PROCESSED:   {PROCESSED}")
print(f"GENERATIONS: {GENERATIONS}")
print(f"RESULTS:     {RESULTS}")

# Verify Notebook 01 outputs exist
required = ["experiment_sample.csv", "lexicons.json",
            "nyt_baseline_axis1.csv", "nyt_baseline_axis2.csv"]
for f in required:
    p = PROCESSED / f
    assert p.exists(), f"Missing prereq from Notebook 01: {p}"
print("\n✓ All Notebook 01 outputs present.")

In [ ]:

# =============================================================================
# CELL 2: Load Notebook 01 outputs
# =============================================================================

sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")
sampled["pub_date"] = pd.to_datetime(sampled["pub_date"])
print(f"Loaded sample: {len(sampled)} headlines")
print(f"  Business: {(sampled['news_desk'] == 'Business').sum()}")
print(f"  Foreign:  {(sampled['news_desk'] == 'Foreign').sum()}")

with open(PROCESSED / "lexicons.json") as f:
    lex = json.load(f)
BUSINESS_FRAME_V4 = set(lex["BUSINESS_FRAME_V4"])
WORLD_FRAME_V4 = set(lex["WORLD_FRAME_V4"])
print(f"\nLexicons: BIZ {len(BUSINESS_FRAME_V4)} | WORLD {len(WORLD_FRAME_V4)}")

nyt_axis1 = pd.read_csv(PROCESSED / "nyt_baseline_axis1.csv")
nyt_axis2 = pd.read_csv(PROCESSED / "nyt_baseline_axis2.csv")
print(f"NYT baseline: axis1 {len(nyt_axis1)} | axis2 {len(nyt_axis2)}")

In [ ]:

# =============================================================================
# CELL 3: Tokenization helpers (must match Notebook 01)
# =============================================================================

TOK_RE = re.compile(r"[A-Za-z][A-Za-z\-']+")
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "do", "does", "did", "will",
    "would", "could", "should", "may", "might", "must", "can", "of", "to",
    "in", "on", "at", "by", "for", "with", "from", "as", "that", "this",
    "these", "those", "it", "its", "their", "they", "them", "we", "our",
    "us", "you", "your", "i", "my", "me", "he", "his", "him", "she", "her",
    "not", "no", "if", "than", "then", "so", "because", "while", "when",
    "where", "what", "which", "who", "whom", "how", "all", "any", "some",
    "more", "most", "other", "such", "only", "own", "same", "very", "just",
    "also", "still", "yet", "even", "now", "after", "before", "during",
    "between", "through", "over", "under", "above", "below", "out", "up",
    "down", "off", "into", "onto", "about",
}


def tokens(text):
    if not isinstance(text, str):
        return []
    return [t.lower() for t in TOK_RE.findall(text)
            if t.lower() not in STOPWORDS and len(t) > 2]


def entity_hit_rates(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0, 0.0, 0, 0
    toks = tokens(text)
    if not toks:
        return 0.0, 0.0, 0, 0
    biz_n = sum(1 for t in toks if t in BUSINESS_FRAME_V4)
    world_n = sum(1 for t in toks if t in WORLD_FRAME_V4)
    return (
        100.0 * biz_n / len(toks),
        100.0 * world_n / len(toks),
        biz_n,
        world_n,
    )

In [ ]:
# Set your OpenAI API key in the environment BEFORE running (do NOT hardcode):
#   export OPENAI_API_KEY="sk-..."
import os
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment first."

In [ ]:

# =============================================================================
# CELL 4: OpenAI API caller
# =============================================================================

from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), \
    "OPENAI_API_KEY not set. Run: os.environ['OPENAI_API_KEY'] = 'sk-...'"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

OPENAI_MODEL = "gpt-4o-mini"
MAX_OUTPUT_TOKENS = 3200


def call_openai(system, user, model=OPENAI_MODEL,
                temperature=0.7, max_output_tokens=MAX_OUTPUT_TOKENS,
                retries=4, sleep_seconds=8):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = client.responses.create(
                model=model,
                instructions=system,
                input=user,
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = r.output_text
            if text is None or not str(text).strip():
                raise RuntimeError(f"Empty response. id={r.id}")
            return text.strip()
        except Exception as e:
            last_err = e
            print(f"  [retry {attempt}/{retries}] {type(e).__name__}: {e}")
            time.sleep(sleep_seconds * attempt)
    raise RuntimeError(f"OpenAI failed after {retries} attempts: {last_err}")


# Smoke test
print("Testing OpenAI...")
print(call_openai("You are a helpful assistant.", "Reply with exactly: pong",
                  temperature=0, max_output_tokens=20))  # 16+ is required

In [ ]:


# =============================================================================
# CELL 5: Define the four prompts
# =============================================================================

# ---- V.A — current / reconstruction ----
SYSTEM_A = (
    "You are a staff journalist writing for The New York Times. "
    "You write in standard NYT register: clear lede, supporting paragraphs "
    "with context and attribution, sourcing to named or general authorities "
    "(e.g., 'analysts said', 'according to company filings'). "
    "Do not fabricate direct quotes from named individuals you cannot verify. "
    "Write only the article body — no headline, no byline, no editor's note."
)
PROMPT_A = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Begin the article now."""


# ---- V.B — minimal anchor (no NYT brand) ----
SYSTEM_B = "You are writing a news article. Output only the article body."
PROMPT_B = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write a full-length news article body under this headline, in this section, on this date. Write from the perspective of the publication date.

Target length: approximately 900 to 1,300 words. Do not include a headline, byline, bullet points, or editor's note."""


# ---- V.D — anti-brochure (NYT system + style constraints) ----
SYSTEM_D = SYSTEM_A
PROMPT_D = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Additional style constraints:
- Avoid promotional, celebratory, or corporate brochure-like language.
- Do not frame the story as a corporate announcement, market opportunity, or institutional success story unless the headline clearly requires it.
- Lead with concrete stakes, tensions, constraints, risks, or consequences rather than generic background or scene-setting.
- Assume readers already know the basic context about the main entities involved; avoid over-explaining.
- Use sober, direct, plain language.

Begin the article now."""


# ---- V.E — explicit institutional frame + anti-brochure ----
SYSTEM_E = SYSTEM_A
PROMPT_E = """SECTION: {section} desk
DATE OF PUBLICATION: {date}
HEADLINE: "{headline}"

Write the complete news article as it would have appeared in print on this date, in this section, under this headline. Target length is roughly 900 to 1,300 words, with a proper lede and several supporting paragraphs. Write from the knowledge and perspective available at the time of publication.

Use the section-specific frame as the primary organizing logic of the article, while adapting it to the specific headline:
- For Business desk articles, foreground firms, markets, investors, regulation, supply chains, corporate strategy, economic uncertainty, and commercial or technological risk.
- For World desk articles, foreground states, diplomacy, security, governance, rights, sovereignty, international institutions, and geopolitical consequences.

Additional style constraints:
- Avoid promotional, celebratory, or corporate brochure-like language.
- Do not frame the story as a corporate announcement, market opportunity, or institutional success story unless the headline explicitly requires it.
- Lead with concrete stakes, tensions, constraints, risks, or consequences rather than generic background or scene-setting.
- Assume readers already know the basic context about the main entities involved; avoid over-explaining.
- Use sober, direct, plain language.

Begin the article now."""

PROMPTS = {
    "A": (SYSTEM_A, PROMPT_A),
    "B": (SYSTEM_B, PROMPT_B),
    "D": (SYSTEM_D, PROMPT_D),
    "E": (SYSTEM_E, PROMPT_E),
}
print(f"Defined {len(PROMPTS)} prompts: {list(PROMPTS.keys())}")

In [ ]:


# =============================================================================
# CELL 6: Generate articles (len(sampled) x REPS x prompts) with checkpoint
# =============================================================================

REPS = 3
GEN_FILE = GENERATIONS / "generations_gpt.csv"

if GEN_FILE.exists():
    df_gen = pd.read_csv(GEN_FILE)
    done = set(zip(df_gen["sample_id"], df_gen["rep"], df_gen["prompt_version"]))
    rows = df_gen.to_dict("records")
    print(f"Resuming: {len(done)} generations already saved.")
else:
    rows = []
    done = set()

saved_since = 0
SAVE_EVERY = 10

target_calls = len(PROMPTS) * len(sampled) * REPS  # prompts x headlines x reps
print(f"Target: {target_calls} total calls. Already done: {len(done)}. To do: {target_calls - len(done)}")

for prompt_name, (sys_msg, user_template) in PROMPTS.items():
    for _, r in sampled.iterrows():
        sid = r["sample_id"]
        section = "Business" if r["news_desk"] == "Business" else "World"
        date_str = r["pub_date"].strftime("%B %d, %Y").replace(" 0", " ")
        prompt = user_template.format(
            section=section, date=date_str, headline=r["headline"],
        )
        for rep in range(REPS):
            key = (sid, rep, prompt_name)
            if key in done:
                continue
            print(f"[{prompt_name}] sid={sid} {section[:3]} rep={rep+1}/{REPS}  "
                  f"{r['headline'][:60]}...")
            try:
                text = call_openai(sys_msg, prompt)
                wc = len(text.split())
                print(f"    → {wc} words")
            except Exception as e:
                print(f"  !! {type(e).__name__}: {e}")
                text = f"[ERROR: {type(e).__name__}]"
                wc = 0

            rows.append({
                "sample_id": sid,
                "rep": rep,
                "prompt_version": prompt_name,
                "headline": r["headline"],
                "news_desk": r["news_desk"],
                "section_prompted": section,
                "pub_date": str(r["pub_date"]),
                "generated_full_text": text,
                "generated_word_count": wc,
                "model": OPENAI_MODEL,
            })
            saved_since += 1

            if saved_since >= SAVE_EVERY:
                pd.DataFrame(rows).to_csv(GEN_FILE, index=False)
                print(f"    ✓ checkpoint saved ({len(rows)} rows)")
                saved_since = 0

            time.sleep(0.5)

# Final save
df_gen = pd.DataFrame(rows)
df_gen.to_csv(GEN_FILE, index=False)
print(f"\nTotal generations: {len(df_gen)}")
print(f"Saved: {GEN_FILE}")

ok = df_gen[~df_gen["generated_full_text"].astype(str).str.startswith("[ERROR")]
print(f"Successful: {len(ok)}")
print("\nWord counts by prompt x desk:")
print(ok.groupby(["prompt_version", "news_desk"])["generated_word_count"].describe().round(0))

In [ ]:


# =============================================================================
# CELL 7: Load DistilBERT for sentiment scoring (same config as Notebook 01)
# =============================================================================

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_TOKENS_PER_CHUNK = 450
STRIDE = 50
BATCH_SIZE = 8

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_model.to(device)
sentiment_model.eval()


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def make_token_chunks(text, max_tokens=MAX_TOKENS_PER_CHUNK, stride=STRIDE):
    if not isinstance(text, str) or not text.strip():
        return []
    token_ids = tokenizer(
        text, add_special_tokens=False,
        truncation=False, return_attention_mask=False,
    )["input_ids"]
    if len(token_ids) == 0:
        return []
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + max_tokens
        chunk_ids = [tokenizer.cls_token_id] + token_ids[start:end] + [tokenizer.sep_token_id]
        chunks.append(chunk_ids)
        if end >= len(token_ids):
            break
        start = end - stride
    return chunks


def predict_token_chunks(token_id_chunks, batch_size=BATCH_SIZE):
    if not token_id_chunks:
        return pd.DataFrame(columns=["chunk_id", "neg_prob", "pos_prob", "sentiment_score"])
    rows_out = []
    for start in range(0, len(token_id_chunks), batch_size):
        batch = token_id_chunks[start:start + batch_size]
        max_len = max(len(x) for x in batch)
        pad_id = tokenizer.pad_token_id
        input_ids, attn = [], []
        for ids in batch:
            pad_n = max_len - len(ids)
            input_ids.append(ids + [pad_id] * pad_n)
            attn.append([1] * len(ids) + [0] * pad_n)
        inputs = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long).to(device),
            "attention_mask": torch.tensor(attn, dtype=torch.long).to(device),
        }
        with torch.inference_mode():
            out = sentiment_model(**inputs)
            probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()
        for j, prob in enumerate(probs):
            neg, pos = float(prob[0]), float(prob[1])
            rows_out.append({
                "chunk_id": start + j,
                "neg_prob": neg,
                "pos_prob": pos,
                "sentiment_score": pos - neg,
            })
        clear_memory()
    return pd.DataFrame(rows_out)

In [ ]:


# =============================================================================
# CELL 8: Score all GPT generations on entity + sentiment axes
# =============================================================================

axis1_rows = []
axis2_rows = []

print(f"Scoring {len(df_gen)} GPT generations on both axes...")
for _, r in tqdm(df_gen.iterrows(), total=len(df_gen)):
    if str(r["generated_full_text"]).startswith("[ERROR"):
        continue
    text = r["generated_full_text"]

    # Axis 1: entity
    br, wr, bh, wh = entity_hit_rates(text)
    axis1_rows.append({
        "sample_id": r["sample_id"],
        "rep": r["rep"],
        "prompt_version": r["prompt_version"],
        "news_desk": r["news_desk"],
        "model": r["model"],
        "biz_rate": br,
        "world_rate": wr,
        "biz_hits_n": bh,
        "world_hits_n": wh,
        "word_count": r["generated_word_count"],
    })

    # Axis 2: sentiment
    try:
        chunks = make_token_chunks(text)
        chunk_df = predict_token_chunks(chunks)
        if len(chunk_df) > 0:
            axis2_rows.append({
                "sample_id": r["sample_id"],
                "rep": r["rep"],
                "prompt_version": r["prompt_version"],
                "news_desk": r["news_desk"],
                "model": r["model"],
                "sentiment": chunk_df["sentiment_score"].mean(),
                "n_chunks": len(chunk_df),
                "word_count": r["generated_word_count"],
            })
    except Exception as e:
        print(f"Sentiment error sid={r['sample_id']} {r['prompt_version']} rep={r['rep']}: {e}")

axis1_gpt = pd.DataFrame(axis1_rows)
axis2_gpt = pd.DataFrame(axis2_rows)
axis1_gpt.to_csv(RESULTS / "axis1_gpt.csv", index=False)
axis2_gpt.to_csv(RESULTS / "axis2_gpt.csv", index=False)
print(f"\nSaved: axis1_gpt.csv ({len(axis1_gpt)} rows)")
print(f"Saved: axis2_gpt.csv ({len(axis2_gpt)} rows)")

In [ ]:


# =============================================================================
# CELL 9: Main results — center, gap, fidelity per prompt
# =============================================================================

# Combine NYT baseline and GPT generations into one "all" frame for comparison
nyt_axis1_full = nyt_axis1.copy()
nyt_axis1_full["prompt_version"] = "NYT"
nyt_axis1_full["rep"] = -1
nyt_axis1_full["model"] = "NYT (real)"

nyt_axis2_full = nyt_axis2.copy()
nyt_axis2_full["prompt_version"] = "NYT"
nyt_axis2_full["rep"] = -1
nyt_axis2_full["model"] = "NYT (real)"

cols_a1 = ["sample_id", "rep", "prompt_version", "news_desk", "model",
           "biz_rate", "world_rate", "word_count"]
cols_a2 = ["sample_id", "rep", "prompt_version", "news_desk", "model",
           "sentiment", "word_count"]

all_axis1 = pd.concat([nyt_axis1_full[cols_a1], axis1_gpt[cols_a1]], ignore_index=True)
all_axis2 = pd.concat([nyt_axis2_full[cols_a2], axis2_gpt[cols_a2]], ignore_index=True)

# ---- Sentiment summary (center / gap) ----
print("=" * 78)
print("SENTIMENT — CENTER and GAP per prompt")
print("=" * 78)
print(f"  Center = (Business mean + Foreign mean) / 2")
print(f"  Gap    = Business mean - Foreign mean")
print()

summary = []
for v in ["NYT", "A", "B", "D", "E"]:
    biz = all_axis2[(all_axis2["prompt_version"] == v) & (all_axis2["news_desk"] == "Business")]["sentiment"]
    foreign = all_axis2[(all_axis2["prompt_version"] == v) & (all_axis2["news_desk"] == "Foreign")]["sentiment"]
    if not (len(biz) and len(foreign)):
        continue
    center = (biz.mean() + foreign.mean()) / 2
    gap = biz.mean() - foreign.mean()
    direction = "matches paradox" if gap < 0 else ("REVERSED" if gap > 0.01 else "FLAT")
    print(f"  {v:4s}  n_biz={len(biz):3d} n_for={len(foreign):3d}  "
          f"Biz={biz.mean():+.4f}  For={foreign.mean():+.4f}  "
          f"Gap={gap:+.4f} ({direction})  Center={center:+.4f}")
    summary.append({
        "prompt": v, "n_biz": len(biz), "n_foreign": len(foreign),
        "biz_mean": biz.mean(), "foreign_mean": foreign.mean(),
        "biz_std": biz.std() if len(biz) > 1 else None,
        "foreign_std": foreign.std() if len(foreign) > 1 else None,
        "center": center, "gap": gap,
    })

# ---- Entity fidelity summary ----
print("\n" + "=" * 78)
print("ENTITY — FIDELITY per prompt")
print("=" * 78)
print()
fidelity_rows = []
for v in ["NYT", "A", "B", "D", "E"]:
    biz_sub = all_axis1[(all_axis1["prompt_version"] == v) & (all_axis1["news_desk"] == "Business")]
    for_sub = all_axis1[(all_axis1["prompt_version"] == v) & (all_axis1["news_desk"] == "Foreign")]
    if not (len(biz_sub) and len(for_sub)):
        continue
    biz_fid = (biz_sub["biz_rate"] - biz_sub["world_rate"]).mean()
    for_fid = (for_sub["world_rate"] - for_sub["biz_rate"]).mean()
    print(f"  {v:4s}  Biz_fid = {biz_fid:+.3f}   For_fid = {for_fid:+.3f}")
    fidelity_rows.append({
        "prompt": v, "biz_fidelity": biz_fid, "for_fidelity": for_fid,
    })

main_summary_df = pd.DataFrame(summary).merge(
    pd.DataFrame(fidelity_rows), on="prompt"
)
main_summary_df.to_csv(RESULTS / "results_main_summary.csv", index=False)
print(f"\nSaved: results_main_summary.csv")


In [ ]:

# =============================================================================
# CELL 10: Robustness 1 — headline-aggregated stats (fix pseudo-replication)
# =============================================================================

print("=" * 78)
print("ROBUSTNESS 1: HEADLINE-LEVEL AGGREGATION")
print("=" * 78)
print("Average reps within each (sample_id, prompt) before computing center/gap.\n")

agg_axis2 = all_axis2.groupby(
    ["sample_id", "prompt_version", "news_desk"], as_index=False
)["sentiment"].mean()

agg_summary = []
for v in ["NYT", "A", "B", "D", "E"]:
    biz = agg_axis2[(agg_axis2["prompt_version"] == v) & (agg_axis2["news_desk"] == "Business")]["sentiment"]
    foreign = agg_axis2[(agg_axis2["prompt_version"] == v) & (agg_axis2["news_desk"] == "Foreign")]["sentiment"]
    if not (len(biz) and len(foreign)):
        continue
    center = (biz.mean() + foreign.mean()) / 2
    gap = biz.mean() - foreign.mean()
    print(f"  {v:4s}  n_biz={len(biz)} n_for={len(foreign)}  "
          f"Biz={biz.mean():+.4f}±{(biz.std()/np.sqrt(len(biz))):.4f}  "
          f"For={foreign.mean():+.4f}±{(foreign.std()/np.sqrt(len(foreign))):.4f}  "
          f"Gap={gap:+.4f}  Center={center:+.4f}")
    agg_summary.append({
        "prompt": v, "n_biz": len(biz), "n_foreign": len(foreign),
        "biz_mean": biz.mean(), "biz_se": biz.std() / np.sqrt(len(biz)) if len(biz) > 1 else 0,
        "foreign_mean": foreign.mean(), "foreign_se": foreign.std() / np.sqrt(len(foreign)) if len(foreign) > 1 else 0,
        "center": center, "gap": gap,
    })

pd.DataFrame(agg_summary).to_csv(RESULTS / "results_headline_aggregated.csv", index=False)
print(f"\nSaved: results_headline_aggregated.csv")

In [ ]:

# =============================================================================
# CELL 11: Robustness 2 — length confound check
# =============================================================================

print("\n" + "=" * 78)
print("ROBUSTNESS 2: WORD COUNT CONFOUND CHECK")
print("=" * 78)

length_df = axis2_gpt[["sample_id", "rep", "prompt_version", "news_desk",
                       "sentiment", "word_count"]].copy()

# Overall correlation
overall_r = length_df[["word_count", "sentiment"]].corr().iloc[0, 1]
print(f"\nOverall correlation r(word_count, sentiment) = {overall_r:+.3f}")
if abs(overall_r) < 0.2:
    print("  → Weak. Length is NOT a confounder.")
elif abs(overall_r) < 0.4:
    print("  → Moderate. Length may be a partial confounder.")
else:
    print("  → Strong. Length IS a confounder; report length-stratified results.")

# Per-prompt correlation
print("\nPer-prompt correlations:")
for v in ["A", "B", "D", "E"]:
    sub = length_df[length_df["prompt_version"] == v]
    if len(sub) >= 3:
        r = sub[["word_count", "sentiment"]].corr().iloc[0, 1]
        print(f"  {v}: n={len(sub)}, r={r:+.3f}")

# Length-bin stratified means
def length_bin(wc):
    if wc < 800:
        return "short_<800"
    elif wc < 1000:
        return "medium_800_999"
    else:
        return "long_>=1000"

length_df["length_bin"] = length_df["word_count"].apply(length_bin)
print("\nLength-stratified sentiment by prompt × bin:")
length_summary = length_df.groupby(["length_bin", "prompt_version"]).agg(
    n=("sentiment", "count"),
    sentiment_mean=("sentiment", "mean"),
    sentiment_std=("sentiment", "std"),
    avg_words=("word_count", "mean"),
).round(4).reset_index()
print(length_summary.to_string(index=False))

length_summary.to_csv(RESULTS / "results_robustness_length.csv", index=False)
print(f"\nSaved: results_robustness_length.csv")

In [ ]:

# =============================================================================
# CELL 12: Final summary
# =============================================================================

print("\n" + "=" * 78)
print("NOTEBOOK 02 COMPLETE")
print("=" * 78)

for f in sorted(GENERATIONS.glob("*.csv")) + sorted(RESULTS.glob("*.csv")):
    size = f.stat().st_size
    print(f"  {f.relative_to(ROOT)}  {size:,} bytes")

print("\n→ Notebook 03 will run Gemini robustness on the same headlines.")


In [ ]:
# =============================================================================
# CELL 12.5: Reload state from disk (in case kernel was restarted)
# This makes section 13-17 self-sufficient - no dependency on earlier cells.
# =============================================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from collections import Counter

# If ROOT/PATHS aren't set (kernel restart), set them (same convention as CELL 1)
if "RESULTS" not in dir():
    ROOT = Path("..").resolve()
    PROCESSED = ROOT / "data" / "processed"
    GENERATIONS = ROOT / "data" / "generations"
    RESULTS = ROOT / "data" / "results"

# Reload all CSVs needed for sections 13-17
sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")
nyt_axis2 = pd.read_csv(PROCESSED / "nyt_baseline_axis2.csv")
axis2_gpt = pd.read_csv(RESULTS / "axis2_gpt.csv")
df_gen = pd.read_csv(GENERATIONS / "generations_gpt.csv")

# Reload lexicons
with open(PROCESSED / "lexicons.json") as f:
    lex = json.load(f)
BUSINESS_FRAME_V4 = set(lex["BUSINESS_FRAME_V4"])
WORLD_FRAME_V4 = set(lex["WORLD_FRAME_V4"])

# Re-define tokenizer helper (used by lexical fingerprinting)
TOK_RE = re.compile(r"[A-Za-z][A-Za-z\-']+")
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "do", "does", "did", "will",
    "would", "could", "should", "may", "might", "must", "can", "of", "to",
    "in", "on", "at", "by", "for", "with", "from", "as", "that", "this",
    "these", "those", "it", "its", "their", "they", "them", "we", "our",
    "us", "you", "your", "i", "my", "me", "he", "his", "him", "she", "her",
    "not", "no", "if", "than", "then", "so", "because", "while", "when",
    "where", "what", "which", "who", "whom", "how", "all", "any", "some",
    "more", "most", "other", "such", "only", "own", "same", "very", "just",
    "also", "still", "yet", "even", "now", "after", "before", "during",
    "between", "through", "over", "under", "above", "below", "out", "up",
    "down", "off", "into", "onto", "about",
}

def tokens(text):
    if not isinstance(text, str):
        return []
    return [t.lower() for t in TOK_RE.findall(text)
            if t.lower() not in STOPWORDS and len(t) > 2]

print(f"✓ Loaded sampled:    {len(sampled)} rows")
print(f"✓ Loaded nyt_axis2:  {len(nyt_axis2)} rows")
print(f"✓ Loaded axis2_gpt:  {len(axis2_gpt)} rows")
print(f"✓ Loaded df_gen:     {len(df_gen)} rows")
print(f"✓ Loaded lexicons:   BIZ {len(BUSINESS_FRAME_V4)} | WORLD {len(WORLD_FRAME_V4)}")

In [ ]:
# =============================================================================
# CELL 13: Per-headline LLM - NYT sentiment difference
# Each headline becomes a paired comparison unit, eliminating headline charge.
# =============================================================================

print("=" * 78)
print("ANALYSIS 1 — PER-HEADLINE LLM-NYT DEVIATION")
print("=" * 78)
print()
print("For each headline, compute (LLM_sentiment - NYT_sentiment).")
print("Aggregate over reps within each (headline, prompt) cell.")
print()

# Aggregate GPT reps within (sample_id, prompt) to one value per headline-prompt
gpt_per_headline = axis2_gpt.groupby(
    ["sample_id", "prompt_version", "news_desk"], as_index=False
)["sentiment"].mean()

# NYT baseline (one value per headline)
nyt_per_headline = nyt_axis2[["sample_id", "news_desk", "sentiment"]].rename(
    columns={"sentiment": "nyt_sentiment"}
)

# Merge: per headline, GPT score and NYT score side by side
merged = gpt_per_headline.merge(
    nyt_per_headline[["sample_id", "nyt_sentiment"]],
    on="sample_id"
)
merged["deviation"] = merged["sentiment"] - merged["nyt_sentiment"]

# Summary by prompt × desk
print("Mean deviation (LLM - NYT) by prompt × desk:")
print()
deviation_summary = merged.groupby(
    ["prompt_version", "news_desk"]
).agg(
    n=("deviation", "count"),
    deviation_mean=("deviation", "mean"),
    deviation_std=("deviation", "std"),
    deviation_median=("deviation", "median"),
).round(4).reset_index()
print(deviation_summary.to_string(index=False))
deviation_summary.to_csv(RESULTS / "results_per_headline_deviation.csv", index=False)
print(f"\nSaved: results_per_headline_deviation.csv")

# Across desks: which prompt minimizes total deviation magnitude?
print("\n" + "=" * 78)
print("Mean |deviation| by prompt (lower = closer to NYT):")
print("=" * 78)
abs_dev = merged.copy()
abs_dev["abs_deviation"] = abs_dev["deviation"].abs()
abs_summary = abs_dev.groupby("prompt_version")["abs_deviation"].agg(["mean", "std", "count"]).round(4)
print(abs_summary)

In [ ]:

# =============================================================================
# CELL 14: Paired Wilcoxon — does each prompt produce sentiment significantly
# different from NYT, when paired by headline?
# =============================================================================

print("\n" + "=" * 78)
print("ANALYSIS 2 — PAIRED WILCOXON: GPT vs NYT (within-headline)")
print("=" * 78)
print()
print("For each prompt × desk, test whether the per-headline deviation is")
print("significantly different from zero (meaning GPT differs from NYT).")
print("Paired test eliminates headline-level variance.\n")

from scipy.stats import wilcoxon

paired_results = []
for v in ["A", "B", "D", "E"]:
    for desk in ["Business", "Foreign"]:
        sub = merged[(merged["prompt_version"] == v) & (merged["news_desk"] == desk)]
        if len(sub) < 5:
            continue
        # Paired comparison: LLM_sentiment vs NYT_sentiment for same headlines
        try:
            stat, p = wilcoxon(
                sub["sentiment"].values,
                sub["nyt_sentiment"].values,
                alternative="two-sided"
            )
        except Exception as e:
            stat, p = np.nan, np.nan
        sig = "✓ p<0.05" if p < 0.05 else "✗ n.s."
        print(f"  {v} {desk:9s}  n={len(sub):2d}  "
              f"GPT={sub['sentiment'].mean():+.4f}  NYT={sub['nyt_sentiment'].mean():+.4f}  "
              f"deviation={sub['deviation'].mean():+.4f}  W={stat:.1f} p={p:.4f}  {sig}")
        paired_results.append({
            "prompt": v, "desk": desk, "n": len(sub),
            "gpt_mean": sub["sentiment"].mean(),
            "nyt_mean": sub["nyt_sentiment"].mean(),
            "deviation_mean": sub["deviation"].mean(),
            "wilcoxon_stat": stat, "p_value": p,
        })

pd.DataFrame(paired_results).to_csv(RESULTS / "results_paired_wilcoxon.csv", index=False)
print(f"\nSaved: results_paired_wilcoxon.csv")

# Cross-prompt comparison: is one prompt closer to NYT than another?
print("\n" + "=" * 78)
print("Pairwise: does one prompt produce smaller deviation than another?")
print("(paired by headline, |deviation_promptA| vs |deviation_promptB|)")
print("=" * 78)
print()
prompt_pairs = [("A", "B"), ("A", "D"), ("A", "E"), ("B", "D"), ("B", "E"), ("D", "E")]
for p1, p2 in prompt_pairs:
    pivot = merged[merged["prompt_version"].isin([p1, p2])].pivot_table(
        index="sample_id", columns="prompt_version", values="deviation"
    ).dropna()
    if len(pivot) < 5:
        continue
    abs1 = pivot[p1].abs().values
    abs2 = pivot[p2].abs().values
    try:
        stat, p = wilcoxon(abs1, abs2, alternative="two-sided")
    except Exception:
        stat, p = np.nan, np.nan
    direction = (
        f"{p1} closer to NYT" if abs1.mean() < abs2.mean() else f"{p2} closer to NYT"
    )
    sig = "✓" if p < 0.05 else " "
    print(f"  {p1} vs {p2}  n={len(pivot)}  "
          f"|dev_{p1}|={abs1.mean():.3f}  |dev_{p2}|={abs2.mean():.3f}  "
          f"p={p:.4f}  {direction} {sig}")

In [ ]:

# =============================================================================
# CELL 15: Cross-prompt consistency within each headline
# Are prompt effects systematic, or does one prompt "win" on some headlines
# and lose on others?
# =============================================================================

print("\n" + "=" * 78)
print("ANALYSIS 3 — CROSS-PROMPT WITHIN-HEADLINE CONSISTENCY")
print("=" * 78)
print()
print("For each headline, rank the 4 prompts by |deviation from NYT|.")
print("If E always ranks #1, prompt effect is consistent.")
print("If rankings vary across headlines, prompt effect is topic-dependent.\n")

# Per-headline ranking
ranking_rows = []
for sid in merged["sample_id"].unique():
    sub = merged[merged["sample_id"] == sid]
    sub_with_abs = sub.copy()
    sub_with_abs["abs_dev"] = sub_with_abs["deviation"].abs()
    sub_sorted = sub_with_abs.sort_values("abs_dev")
    sub_sorted["rank"] = range(1, len(sub_sorted) + 1)
    headline = sampled[sampled["sample_id"] == sid]["headline"].iloc[0]
    desk = sampled[sampled["sample_id"] == sid]["news_desk"].iloc[0]
    for _, r in sub_sorted.iterrows():
        ranking_rows.append({
            "sample_id": sid,
            "headline": headline[:60],
            "desk": desk,
            "prompt": r["prompt_version"],
            "deviation": r["deviation"],
            "abs_deviation": r["abs_dev"],
            "rank": r["rank"],
        })
ranking_df = pd.DataFrame(ranking_rows)

# Mean rank per prompt (1 = closest to NYT, 4 = farthest)
print("Mean rank per prompt (1 = always closest to NYT, 4 = always farthest):")
mean_ranks = ranking_df.groupby("prompt")["rank"].agg(["mean", "std", "count"]).round(3)
print(mean_ranks)
ranking_df.to_csv(RESULTS / "results_cross_prompt_ranks.csv", index=False)
print(f"\nSaved: results_cross_prompt_ranks.csv")

# How often does each prompt achieve rank 1?
print("\nRank-1 frequency (how often each prompt is closest-to-NYT for a headline):")
rank1_freq = ranking_df[ranking_df["rank"] == 1].groupby("prompt").size()
total_headlines = ranking_df["sample_id"].nunique()
for p in ["A", "B", "D", "E"]:
    n = rank1_freq.get(p, 0)
    print(f"  {p}: {n}/{total_headlines} headlines ({100*n/total_headlines:.0f}%)")

In [ ]:

# =============================================================================
# CELL 16: Lexical fingerprinting — which words does GPT overuse vs NYT?
# This is the mechanism-level finding: explains *why* the offset exists.
# =============================================================================

print("\n" + "=" * 78)
print("ANALYSIS 4 — LEXICAL FINGERPRINTING")
print("=" * 78)
print()
print("Find words that GPT articles use much more than NYT articles, and vice versa.")
print()

# Build NYT corpus from full_text in sampled
nyt_full_texts = sampled.set_index("sample_id")["full_text"].to_dict()

# Build GPT corpus by prompt
gpt_corpus_by_prompt = {}
for v in ["A", "B", "D", "E"]:
    sub = df_gen[df_gen["prompt_version"] == v]
    sub = sub[~sub["generated_full_text"].astype(str).str.startswith("[ERROR")]
    gpt_corpus_by_prompt[v] = sub["generated_full_text"].tolist()

nyt_corpus_full = list(nyt_full_texts.values())


def aggregate_word_frequency(corpus_list):
    """Return Counter of word frequencies normalized by corpus size."""
    all_tokens = [tok for text in corpus_list for tok in tokens(text)]
    total = len(all_tokens)
    if total == 0:
        return Counter(), 0
    return Counter(all_tokens), total


nyt_freq, nyt_total = aggregate_word_frequency(nyt_corpus_full)


def find_overused_underused(gpt_corpus, prompt_label, top_n=20):
    """Return top words GPT overuses vs NYT, and top words GPT underuses."""
    gpt_freq, gpt_total = aggregate_word_frequency(gpt_corpus)
    if gpt_total == 0:
        return [], []

    # Compute log ratio for each word
    all_words = set(gpt_freq.keys()) | set(nyt_freq.keys())
    results = []
    for w in all_words:
        gpt_n = gpt_freq.get(w, 0)
        nyt_n = nyt_freq.get(w, 0)
        # require minimum frequency
        if gpt_n + nyt_n < 10:
            continue
        # smooth with +1
        gpt_rate = (gpt_n + 1) / gpt_total
        nyt_rate = (nyt_n + 1) / nyt_total
        log_ratio = np.log(gpt_rate / nyt_rate)
        results.append((w, log_ratio, gpt_n, nyt_n,
                        round(gpt_rate * 1000, 2),
                        round(nyt_rate * 1000, 2)))
    results.sort(key=lambda x: x[1], reverse=True)
    overused = results[:top_n]
    underused = results[-top_n:][::-1]
    return overused, underused


lexical_summary = []
for v in ["A", "B", "D", "E"]:
    print(f"\n--- Prompt {v} ---")
    overused, underused = find_overused_underused(gpt_corpus_by_prompt[v], v, top_n=20)
    print(f"\n  Top 20 OVERUSED words (GPT >> NYT):")
    print(f"  {'word':<20s}  log_ratio  gpt_freq  nyt_freq  gpt/1k  nyt/1k")
    for w, lr, gn, nn, gpr, npr in overused:
        print(f"  {w:<20s}  {lr:+.3f}     {gn:>6d}   {nn:>6d}   {gpr:6.2f}  {npr:6.2f}")
        lexical_summary.append({
            "prompt": v, "type": "overused", "word": w,
            "log_ratio": lr, "gpt_freq": gn, "nyt_freq": nn,
        })
    print(f"\n  Top 20 UNDERUSED words (GPT << NYT):")
    print(f"  {'word':<20s}  log_ratio  gpt_freq  nyt_freq  gpt/1k  nyt/1k")
    for w, lr, gn, nn, gpr, npr in underused:
        print(f"  {w:<20s}  {lr:+.3f}     {gn:>6d}   {nn:>6d}   {gpr:6.2f}  {npr:6.2f}")
        lexical_summary.append({
            "prompt": v, "type": "underused", "word": w,
            "log_ratio": lr, "gpt_freq": gn, "nyt_freq": nn,
        })

pd.DataFrame(lexical_summary).to_csv(RESULTS / "results_lexical_fingerprint.csv", index=False)
print(f"\nSaved: results_lexical_fingerprint.csv")



In [ ]:

# =============================================================================
# CELL 17: Topic stratification - LLM auto-labeling of topic charge
# Each headline is labeled adversarial vs soft by GPT-4o-mini (cached to disk),
# so the analysis covers the full expanded sample, not just the original 20.
# =============================================================================

from tqdm.auto import tqdm

print("\n" + "=" * 78)
print("ANALYSIS 5 - TOPIC STRATIFICATION (LLM-labeled)")
print("=" * 78)
print()

TOPIC_LABEL_FILE = RESULTS / "topic_charge_labels.csv"

TOPIC_SYS = (
    "You are a media-studies annotator. Classify the topic charge of a news "
    "headline about China as exactly one of two labels:\n"
    "  adversarial = confrontation, restriction, accusation, sanctions, "
    "investigation, security/military, repression, censorship, conflict, or "
    "geopolitical antagonism.\n"
    "  soft = culture, achievement, science, sport, humanitarian, human-interest, "
    "cooperative diplomacy, or other non-confrontational coverage.\n"
    "Reply with exactly one lowercase word: adversarial or soft."
)


def classify_topic_llm(headline):
    out = call_openai(TOPIC_SYS, f'HEADLINE: "{headline}"',
                      temperature=0, max_output_tokens=16).strip().lower()
    if "advers" in out:
        return "adversarial"
    if "soft" in out:
        return "soft"
    return "unclassified"


# Use cached labels if present; otherwise label via API and save.
if TOPIC_LABEL_FILE.exists():
    labels = pd.read_csv(TOPIC_LABEL_FILE)
    print(f"Loaded cached topic labels: {len(labels)} rows  ({TOPIC_LABEL_FILE.name})")
else:
    assert "call_openai" in dir(), \
        "call_openai not defined - run CELL 4/5 (OpenAI client) before labeling."
    recs = []
    for _, r in tqdm(sampled.iterrows(), total=len(sampled), desc="Labeling topic charge"):
        recs.append({
            "sample_id": r["sample_id"],
            "headline": r["headline"],
            "news_desk": r["news_desk"],
            "topic_class": classify_topic_llm(r["headline"]),
        })
    labels = pd.DataFrame(recs)
    labels.to_csv(TOPIC_LABEL_FILE, index=False)
    print(f"Saved topic labels: {TOPIC_LABEL_FILE}")

# Attach labels to sampled
sampled = sampled.drop(columns=["topic_class"], errors="ignore").merge(
    labels[["sample_id", "topic_class"]], on="sample_id", how="left"
)
print("\nTopic classification (desk x class):")
print(sampled.groupby(["news_desk", "topic_class"]).size().unstack(fill_value=0))

# Stratified deviation analysis (merged is built in CELL 13)
merged_with_topic = merged.merge(
    sampled[["sample_id", "topic_class", "headline"]], on="sample_id"
)
print("\nMean per-headline deviation (LLM - NYT) by topic class x prompt x desk:\n")
topic_summary = merged_with_topic.groupby(
    ["topic_class", "prompt_version", "news_desk"]
).agg(
    n=("deviation", "count"),
    deviation_mean=("deviation", "mean"),
    gpt_mean=("sentiment", "mean"),
    nyt_mean=("nyt_sentiment", "mean"),
).round(4).reset_index()
print(topic_summary.to_string(index=False))
topic_summary.to_csv(RESULTS / "results_topic_stratified.csv", index=False)
print(f"\nSaved: results_topic_stratified.csv")

# Full label list - spot-check the LLM
print("\n" + "=" * 78)
print("Topic classification of all headlines (please spot-check):")
print("=" * 78)
for _, r in sampled.sort_values(["news_desk", "topic_class"]).iterrows():
    print(f"  [{r['news_desk'][:3]}] [{r['topic_class']:13s}] {r['headline']}")

In [ ]:

# =============================================================================
# CELL 18: Final summary of analyses
# =============================================================================

print("\n" + "=" * 78)
print("SECTION 13-17 ANALYSES COMPLETE")
print("=" * 78)
print()
print("New result files saved:")
for f in sorted((RESULTS).glob("results_*.csv")):
    size = f.stat().st_size
    print(f"  {f.name:50s}  {size:>8,} bytes")

print("""
Key questions answered:

  1. Per-headline LLM-NYT difference: Which prompt closest to NYT after
     controlling for headline charge?  → see results_per_headline_deviation.csv

  2. Paired Wilcoxon: Statistically, does GPT differ from NYT on a per-headline
     basis?  → see results_paired_wilcoxon.csv

  3. Cross-prompt rank: Is the prompt-engineerability hierarchy CONSISTENT
     across headlines?  → see results_cross_prompt_ranks.csv

  4. Lexical fingerprint: WHICH words does GPT overuse/underuse vs NYT?
     → see results_lexical_fingerprint.csv  (this is paper figure material)

  5. Topic stratification: Does the offset differ between adversarial and
     soft topics?  → see results_topic_stratified.csv

For paper:
- Use #1 and #2 to demonstrate the hierarchy after controlling for headline charge
- Use #3 to show consistency
- Use #4 as the mechanism explanation (lexical figure in discussion)
- Use #5 to validate the topic-charge hypothesis
""")


In [ ]:
# =============================================================================
# CELL 17b: Optional human override of LLM topic labels, then re-run stratified.
# Spot-check the list printed above; correct any wrong labels by sample_id here.
# Leave MANUAL_OVERRIDE empty to trust the LLM labels as-is.
# =============================================================================

MANUAL_OVERRIDE = {
    # sample_id: "adversarial" or "soft",
    # e.g. 5: "soft",
}

if MANUAL_OVERRIDE:
    sampled["topic_class"] = sampled.apply(
        lambda r: MANUAL_OVERRIDE.get(r["sample_id"], r["topic_class"]), axis=1
    )
    # Persist corrected labels back to the cache so future runs reuse them
    sampled[["sample_id", "headline", "news_desk", "topic_class"]].to_csv(
        RESULTS / "topic_charge_labels.csv", index=False
    )
    print(f"Applied {len(MANUAL_OVERRIDE)} manual override(s); cache updated.")
else:
    print("No manual overrides; using LLM labels as-is.")

merged_with_topic_v2 = merged.merge(
    sampled[["sample_id", "topic_class", "headline"]], on="sample_id"
)
topic_summary_v2 = merged_with_topic_v2.groupby(
    ["topic_class", "prompt_version", "news_desk"]
).agg(
    n=("deviation", "count"),
    deviation_mean=("deviation", "mean"),
    gpt_mean=("sentiment", "mean"),
    nyt_mean=("nyt_sentiment", "mean"),
).round(4).reset_index()
print("\nFinal topic-stratified analysis:")
print(topic_summary_v2.to_string(index=False))
topic_summary_v2.to_csv(RESULTS / "results_topic_stratified_v2.csv", index=False)
print(f"\nSaved: results_topic_stratified_v2.csv")